In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
import warnings

warnings.filterwarnings('ignore')

def analisar_turnover_e_padroes(diretorio_dados):
    print("=== INÍCIO DA ANÁLISE DE GIRO DE CARTEIRA (MÍNIMA VARIÂNCIA) ===")
    
    caminho_pesos = os.path.join(diretorio_dados, "historico_pesos_l1_5anos_M_min_var.parquet")
    print("1. A carregar o histórico de pesos da otimização defensiva...")
    df_pesos = pd.read_parquet(caminho_pesos)
    
    # Blindagem
    if not isinstance(df_pesos.index, pd.DatetimeIndex):
        coluna_data_pesos = df_pesos.columns[0]
        df_pesos[coluna_data_pesos] = pd.to_datetime(df_pesos[coluna_data_pesos], errors='coerce')
        df_pesos.set_index(coluna_data_pesos, inplace=True)
    df_pesos.index.name = 'Data'
    df_pesos.index = df_pesos.index.normalize()
    if df_pesos.index.tz is not None:
        df_pesos.index = df_pesos.index.tz_localize(None)
    df_pesos = df_pesos.sort_index()
    
    for coluna in df_pesos.columns:
        if df_pesos[coluna].dtype == 'object' or df_pesos[coluna].dtype.name in ['string', 'string[pyarrow]']:
            df_pesos[coluna] = df_pesos[coluna].astype(str).str.replace(',', '.')
        df_pesos[coluna] = pd.to_numeric(df_pesos[coluna], errors='coerce')
    
    print("2. A calcular a magnitude das alterações mensais...")
    giro_capital = (df_pesos.diff().abs().sum(axis=1)) / 2
    giro_capital = giro_capital.fillna(0)
    
    meses_totais = len(giro_capital) - 1
    meses_sem_giro = (giro_capital == 0).sum() - 1 
    media_giro_ativo = giro_capital[giro_capital > 0].mean() if (giro_capital > 0).any() else 0
    
    maiores_giros = giro_capital.sort_values(ascending=False).head(5)
    menores_giros_ativos = giro_capital[giro_capital > 0].sort_values(ascending=True).head(5)
    
    print("3. A desenhar o gráfico de análise cronológica...")
    plt.figure(figsize=(15, 6))
    
    plt.bar(giro_capital.index, giro_capital.values, width=20, color='dodgerblue', alpha=0.7, edgecolor='black')
    
    plt.axhline(media_giro_ativo, color='red', linestyle='--', linewidth=1.5, 
                label=f'Média de Giro Operacional: {media_giro_ativo:.2%}')
    
    plt.title('Evolução do Giro de Capital (Turnover Mensal) - Mínima Variância', fontsize=16)
    plt.xlabel('Cronologia', fontsize=12)
    plt.ylabel('Património Rebalanceado (%)', fontsize=12)
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    plt.axhline(0, color='black', linewidth=1)
    
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    
    caminho_grafico = os.path.join(diretorio_dados, "analise_turnover_mensal_min_var.png")
    plt.savefig(caminho_grafico, dpi=300, bbox_inches='tight')
    plt.close()
    
    print("\n=== PADRÕES DETETADOS NO ALGORITMO ===")
    print(f"Total de meses analisados: {meses_totais}")
    print(f"Meses com Giro Zero (No-Trade Zone funcionou): {meses_sem_giro} meses ({(meses_sem_giro/meses_totais):.1%})")
    print(f"Giro Médio (quando ocorreu rebalanceamento): {media_giro_ativo:.2%}")
    
    print("\n🔥 TOP 5 - MAIORES CHOQUES NA CARTEIRA (Grandes Mudanças):")
    for data, giro in maiores_giros.items():
        print(f" - {data.strftime('%B de %Y')}: {giro:.2%} do património alterado.")
        
    print(f"\nGráfico exportado para: {caminho_grafico}")
    print("======================================================")
    
    return giro_capital

pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
serie_turnover = analisar_turnover_e_padroes(pasta_dados)

=== INÍCIO DA ANÁLISE DE GIRO DE CARTEIRA (MÍNIMA VARIÂNCIA) ===
1. A carregar o histórico de pesos da otimização defensiva...
2. A calcular a magnitude das alterações mensais...
3. A desenhar o gráfico de análise cronológica...

=== PADRÕES DETETADOS NO ALGORITMO ===
Total de meses analisados: 131
Meses com Giro Zero (No-Trade Zone funcionou): 0 meses (0.0%)
Giro Médio (quando ocorreu rebalanceamento): 0.32%

🔥 TOP 5 - MAIORES CHOQUES NA CARTEIRA (Grandes Mudanças):
 - March de 2020: 14.57% do património alterado.
 - May de 2017: 2.09% do património alterado.
 - April de 2020: 1.91% do património alterado.
 - March de 2016: 1.32% do património alterado.
 - June de 2020: 1.13% do património alterado.

Gráfico exportado para: C:\VSCodeWorkspace\TCC_Escrito\data\analise_turnover_mensal_min_var.png
